In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

## Initialise input widgets and fetch parameters

In [0]:
dbutils.widgets.text("catalog_name", "spotify_catalog")
dbutils.widgets.text("adls_storage_container_name", "")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
adls_storage_container_name = dbutils.widgets.get("adls_storage_container_name")

In [0]:
import os
import sys

project_pth = os.path.join(os.getcwd(), '..')
sys.path.append(project_pth)

In [0]:
from utils.transformations import reusable_transformations
transform_obj = reusable_transformations()

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## DimUser

In [0]:
dim_user_pk_list = ['user_id']
target_table_full_name =  f"{catalog_name}.silver.DimUser"

In [0]:
df_user = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimUser")
    
df_user.display()

In [0]:
df_user = df_user.withColumn("user_name", F.upper(F.col("user_name")))
df_user.display()

In [0]:
df_user = transform_obj.dropColumns(df_user, ["_rescued_data"])
df_user.display()

In [0]:
df_user = transform_obj.add_metadata_cols(df_user, dim_user_pk_list)
df_user.display()

In [0]:
if spark.catalog.tableExists(target_table_full_name):
    target_table_delta = DeltaTable.forName(spark, target_table_full_name)

    merge_keys_list = transform_obj.build_pk_join_string(dim_user_pk_list)
    print(f"Merge Keys Comparison List for {target_table_full_name} is {merge_keys_list}")

    prep_merge_column_list_string = transform_obj.build_prep_merge_column_list_string(df_user)
    print(f"Merge Columns List for {target_table_full_name} is {prep_merge_column_list_string}")
    
    print(f"Since {target_table_full_name} table already exists, we are performing merge operation")
    df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/checkpoint") \
    .trigger(availableNow=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/data") \
    .foreachBatch(transform_obj.perform_merge_operation(target_table_delta, merge_keys_list, prep_merge_column_list_string)) \
    .start() \
    .awaitTermination()
else:
    print(f"Creating {target_table_full_name} and loading data")
    df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/checkpoint") \
    .trigger(availableNow=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/data") \
    .toTable(target_table_full_name)

## DimArtist

In [0]:
dim_artist_pk_list = ['artist_id']
target_table_full_name =  f"{catalog_name}.silver.DimArtist"

In [0]:
df_artist = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimArtist")

df_artist.display()

In [0]:
df_artist = df_artist.withColumn("artist_name", F.upper(F.col("artist_name")))
df_artist.display()

In [0]:
df_artist = transform_obj.dropColumns(df_artist, ["_rescued_data"])
df_artist = transform_obj.add_metadata_cols(df_artist, dim_artist_pk_list)
df_artist.display()

In [0]:
if spark.catalog.tableExists(target_table_full_name):
    target_table_delta = DeltaTable.forName(spark, target_table_full_name)

    merge_keys_list = transform_obj.build_pk_join_string(dim_artist_pk_list)
    print(f"Merge Keys Comparison List for {target_table_full_name} is {merge_keys_list}")

    prep_merge_column_list_string = transform_obj.build_prep_merge_column_list_string(df_artist)
    print(f"Merge Columns List for {target_table_full_name} is {prep_merge_column_list_string}")

    print(f"Since {target_table_full_name} table already exists, we are performing merge operation")
    df_artist.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/checkpoint") \
    .trigger(availableNow=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/data") \
    .foreachBatch(transform_obj.perform_merge_operation(target_table_delta, merge_keys_list, prep_merge_column_list_string)) \
    .start() \
    .awaitTermination()
else:
    print(f"Creating {target_table_full_name} and loading data")
    df_artist.writeStream.format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/checkpoint") \
        .trigger(availableNow=True) \
        .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/data") \
        .toTable(target_table_full_name)

## DimTrack

In [0]:
dim_track_pk_list = ['track_id']
target_table_full_name =  f"{catalog_name}.silver.DimTrack"

In [0]:
df_track = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimTrack")

df_track.display()

In [0]:
df_track = df_track.withColumn("durationFlag", F.when(F.col("duration_sec") <= 150, "low").when((F.col("duration_sec") > 150) & (F.col("duration_sec") < 300), "medium").otherwise("high")
)

df_track.display()

In [0]:
df_track = df_track.withColumn("track_name", F.regexp_replace(F.col("track_name"), "-", " "))
df_track.display()

In [0]:
df_track = transform_obj.dropColumns(df_track, ['_rescued_data'])
df_track = transform_obj.add_metadata_cols(df_track, dim_track_pk_list)
df_track.display()

In [0]:
if spark.catalog.tableExists(target_table_full_name):
    target_table_delta = DeltaTable.forName(spark, target_table_full_name)

    merge_keys_list = transform_obj.build_pk_join_string(dim_track_pk_list)
    print(f"Merge Keys Comparison List for {target_table_full_name} is {merge_keys_list}")

    prep_merge_column_list_string = transform_obj.build_prep_merge_column_list_string(df_track)
    print(f"Merge Columns List for {target_table_full_name} is {prep_merge_column_list_string}")

    print(f"Since {target_table_full_name} table already exists, we are performing merge operation")
    df_track.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/checkpoint") \
    .trigger(availableNow=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/data") \
    .foreachBatch(transform_obj.perform_merge_operation(target_table_delta, merge_keys_list, prep_merge_column_list_string)) \
    .start() \
    .awaitTermination()
else:
    print(f"Creating {target_table_full_name} and loading data")
    df_track.writeStream.format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/checkpoint") \
        .trigger(availableNow=True) \
        .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/data") \
        .toTable(target_table_full_name)

## Dim_Date

In [0]:
target_table_full_name = f"{catalog_name}.silver.DimDate"
dim_date_pk_list = ['date_key']

In [0]:
df_date = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimDate/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimDate")

df_date.display()

In [0]:
df_date = transform_obj.dropColumns(df_date, ['_rescued_data'])
df_date = transform_obj.add_metadata_cols(df_date, dim_date_pk_list)
df_date.display()

In [0]:
print(f"Creating or loading data into {target_table_full_name}")
df_date.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimDate/checkpoint") \
    .trigger(availableNow=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimDate/data") \
    .toTable(target_table_full_name)

## FactStream

In [0]:
fact_stream_pk_list = ['stream_id']
target_table_full_name = f"{catalog_name}.silver.FactStream"

In [0]:
df_fact = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/FactStream")

df_fact.display()

In [0]:
df_track = spark.read.table(f"{catalog_name}.silver.dimtrack")

df_joined_fact = df_fact.alias("df_fact").join(df_track.alias("df_track"), F.col("df_fact.track_id") == F.col("df_track.track_id"), how="left") \
          .select("df_fact.*","artist_id")

In [0]:
df_fact = transform_obj.dropColumns(df_fact, ['_rescued_data'])
df_fact = transform_obj.add_metadata_cols(df_fact, fact_stream_pk_list)
df_fact.display()

In [0]:
if spark.catalog.tableExists(target_table_full_name):
    target_table_delta = DeltaTable.forName(spark, target_table_full_name)

    merge_keys_list = transform_obj.build_pk_join_string(fact_stream_pk_list)
    print(f"Merge Keys Comparison List for {target_table_full_name} is {merge_keys_list}")

    prep_merge_column_list_string = transform_obj.build_prep_merge_column_list_string(df_fact)
    print(f"Merge Columns List for {target_table_full_name} is {prep_merge_column_list_string}")

    print(f"Since {target_table_full_name} table already exists, we are performing merge operation")
    df_fact.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/checkpoint") \
    .trigger(availableNow=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/data") \
    .foreachBatch(transform_obj.perform_merge_operation(target_table_delta, merge_keys_list, prep_merge_column_list_string)) \
    .start() \
    .awaitTermination()
else:
    print(f"Creating {target_table_full_name} and loading data")
    df_fact.writeStream.format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/checkpoint") \
        .trigger(availableNow=True) \
        .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/data") \
        .toTable(target_table_full_name)